# One-region stimulus configurations


New stimulus

## Initialization



Load environment variables

<font color=#fda> NOTE: </font> Prior to loading any modules, we need to load the environment variables from the configuration files. This way, we can configure the TVB modules using environment variables.


In [ ]:
from dotenv import dotenv_values, find_dotenv
import os
import sys
from pathlib import Path


# Busqueda de ficheros dotenv
defaults_file = find_dotenv('.env_default', usecwd=True)

if not defaults_file:
    raise FileNotFoundError('Start Jupyter within the repository containing .env_default.')
env_dir = Path(defaults_file).resolve().parent

# Actualización de varaibles de entorno
settings = {**dotenv_values(defaults_file), **dotenv_values(env_dir / '.env')}
settings = {k: v if v is not None else '' for k, v in settings.items() }
os.environ.update(settings)

# Librerías auxiliares
import_dir = Path(settings.get('IMPORT_DIR') or env_dir).expanduser()
if not import_dir.is_absolute():
    import_dir = env_dir / import_dir
sys.path.insert(0, str(import_dir.resolve()))


Modules

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


In [ ]:
from aux.environment import resolve_directory
from aux.tvb.stimulus import _build_pulse, _build_pulse_train
from aux.tvb.model import build_model
from aux.tvb.integrator import build_integrator
from aux.tvb.simulator import build_single_region_simulator
from aux.tvb.results import SimulationMetadata, ModelMetadata, ResultMetadata
from aux.tvb.plotting import plot_results


Console

In [ ]:
from rich.console import Console
pprint = Console(force_terminal=True, force_jupyter=False, width=140).print

pprint(f'The console is "ready" to print messages with rich formatting.')

Set main directory paths

In [ ]:

#ToDo: Cambiar el nombe de las variables de entorno
WORK_DIR = resolve_directory(settings['WORK_DIR'], env_dir)

os.makedirs(WORK_DIR, exist_ok=True)
pprint(f'Working directory: {WORK_DIR}')

## Simulation values


In [ ]:
# 1) Simulation outputs.
OUTPUT_DTYPE = "float32"
OUTPUT_STATES = ["E"]

STATE_UNITS = {
    "E": "kHz", "I": "kHz",
    "C_ee": "kHz²", "C_ei": "kHz²", "C_ii": "kHz²",
    "W_e": "pA", "W_i": "pA", "ou_drift": "N.A.",
}
STIMULUS_UNITS = {"external_input_ex_ex": "kHz"}


In [ ]:
# 1.1) Simulation parameters (same values as 01b).
DURATION_MS = 2000.0
DT_MS = 0.5


In [ ]:
# 1.2) Stimulus configurations; each entry follows the 01a/01b format.
STIMULUS_CONFIGS = {
    "Pulse": {
        "external_input_ex_ex": {"pulse": {
            "baseline_khz": 0.005, "pulse_amplitude_khz": 0.04,
            "pulse_onset_ms": 500.0, "pulse_duration_ms": 800.0,
        }},
    },
    "Periodic": {
        "external_input_ex_ex": {"pulse_train": {
            "baseline_khz": 0.005, "pulse_amplitude_khz": 0.04,
            "pulse_onset_ms": 500.0, "pulse_duration_ms": 120.0,
            "pulse_period_ms": 250.0,
        }},
    },
    "Steps": {
        "external_input_ex_ex": {"step": {
            "baseline_khz": 0.005, "onset_ms": 500.0,
            "points_ms": [0.0, 250.0, 500.0, 850.0],
            "amplitudes_khz": [0.01, 0.04, 0.015, 0.03],
        }},
    },
    "Smooth transitions": {
        "external_input_ex_ex": {"smooth": {
            "baseline_khz": 0.005, "onset_ms": 500.0,
            "points_ms": [200.0, 450.0, 700.0, 1050.0],
            "amplitudes_khz": [0.01, 0.04, 0.015, 0.03],
        }},
    },
    "Random steps": {
        "external_input_ex_ex": {"random_step": {
            "baseline_khz": 0.005, "onset_ms": 500.0,
            "min_duration_ms": 100.0, "max_duration_ms": 250.0,
            "min_amplitude_khz": 0.005, "max_amplitude_khz": 0.04,
            "seed": 42,
        }},
    },
    "Random smooth transitions": {
        "external_input_ex_ex": {"random_smooth": {
            "baseline_khz": 0.005, "onset_ms": 500.0,
            "min_duration_ms": 100.0, "max_duration_ms": 250.0,
            "min_amplitude_khz": 0.005, "max_amplitude_khz": 0.04,
            "seed": 42,
        }},
    },
}



NOISE_CONFIG = {"mean_khz": 0.0, "std_khz": 0.001, "noise_seed": 7}

NOISE_STIMULUS_CONFIG = {
    name: {
        parameter: {
            func: {**values, **NOISE_CONFIG}
            for func, values in definition.items()
            if func in ("pulse", "pulse_train", "step", "smooth",
                        "random_step", "random_smooth")
        }
        for parameter, definition in stimulus.items() 
        if definition is not None 
    }
    for name, stimulus in STIMULUS_CONFIGS.items()
}

pprint('Noise stimulus configuration', NOISE_STIMULUS_CONFIG)

In [ ]:
STIMULUS_CONFIGS = {
    name: {**stimulus, "external_input_in_ex": {"external_input_ex_ex": 1.0}}
    for name, stimulus in STIMULUS_CONFIGS.items()
}

NOISE_STIMULUS_CONFIG = {
    name: {**stimulus, "external_input_in_ex": {"external_input_ex_ex": 1.0}}
    for name, stimulus in NOISE_STIMULUS_CONFIG.items()
}


In [ ]:
# 1.3) Parámetros del modelo Zerlaut-AdEx.
#MODEL_NAME = "ZerlautAdaptationFirstOrder"
#MODEL_NAME = "ZerlautAdaptationSecondOrder"


MODEL_PARAMETERS = {
    # Población y conectividad
    "N_tot": 10000,
    "p_connect_e": 0.05,
    "p_connect_i": 0.05,
    "g": 0.2,
    "K_ext_e": 400,
    "K_ext_i": 35,
    
    # Parametros de membrana
    "g_L": 10.0,
    "C_m": 200.0,
    "E_L_e": -60.0,
    "E_L_i": -65.0,

    # Parametors de adaptación
    "a_e": 4.0,
    "b_e": 60.0,
    "tau_w_e": 200.0,
    "a_i": 0.0,
    "b_i": 20.0,
    "tau_w_i": 100.0,

    # Parametros sinápticos
    "E_e": 0.0,
    "E_i": -80.0,
    "Q_e": 1.0,
    "Q_i": 5.0,
    "tau_e": 5.0,
    "tau_i": 5.0,
    
    "T": 10.0,

    # Parametro de segundo orden (no hay un motivo matemático)
    "S_i": 1, # Escala la entrada remota recibida por la población inhibidora en el modelo de segundo orden.

    # Parametros de ruido
    "weight_noise": 2e-4, #10.5
    "tau_OU": 2, #0.5
}

# Todos los parámetros son de tipo float menos lo que están en esta lista. 
INTEGER_PARAMETERS = {"N_tot", "K_ext_e", "K_ext_i"}


## Build the stimulus


Each named configuration describes one simulation, with the same parameter-to-stimulus mapping as `01a`. `build_stimulus` generates direct signals first, then scales signals from parameters already defined in that simulation. Steps make discrete changes; smooth transitions reach each target through a cubic interpolation. Random variants generate times and amplitudes, then use the corresponding step or smooth function. Gaussian noise is added separately when its standard deviation is positive.


In [ ]:
def _validate_levels(points_ms, amplitudes_khz):
    points = np.asarray(points_ms, dtype=float)
    amplitudes = np.asarray(amplitudes_khz, dtype=float)

    if (points.ndim != 1 or amplitudes.ndim != 1 or len(points) == 0 or len(points) != len(amplitudes)):
        raise ValueError("points_ms and amplitudes_khz need matching, ordered finite values")
    
    return points, amplitudes

def _smoothstep(fraction):
    return fraction * fraction * (3.0 - 2.0 * fraction)


def _build_step(time_ms, baseline_khz, onset_ms, points_ms, amplitudes_khz):
    points, amplitudes = _validate_levels(points_ms, amplitudes_khz)

    signal = np.full(time_ms.shape, baseline_khz, dtype=float)

    # ToDo: Busacar algo más eficiente. Ahora se sobreescriben muchos valores.
    #       Mirar index = np.searchsorted
    for point, amplitude in zip(points, amplitudes):
        signal[time_ms >= onset_ms + point] = amplitude

    return signal


def _build_smooth(time_ms, baseline_khz, onset_ms, points_ms, amplitudes_khz):
    points, amplitudes = _validate_levels(points_ms, amplitudes_khz)
    
    # Permite transicionar suavemente desde el valor de referencia (baseline_khz) hasta el primer valor de amplitud.
    if points[0] > 0:
        points = np.r_[0.0, points]
        amplitudes = np.r_[baseline_khz, amplitudes]

    signal = np.full(time_ms.shape, baseline_khz, dtype=float)
    
    for start, end, a, b in zip(points[:-1], points[1:], amplitudes[:-1], amplitudes[1:]):
        t0, t1 = onset_ms + start, onset_ms + end
        mask = (time_ms >= t0) & (time_ms < t1)
        fraction = (time_ms[mask] - t0) / (t1 - t0)

        signal[mask] = a + (b - a) * _smoothstep(fraction)

    signal[time_ms >= onset_ms + points[-1]] = amplitudes[-1]

    return signal


def _random_stimulation(time_ms, onset_ms, min_duration_ms, max_duration_ms,
                        min_amplitude_khz, max_amplitude_khz, seed=None):
    
    if (min_duration_ms <= 0 or max_duration_ms < min_duration_ms
            or max_amplitude_khz < min_amplitude_khz):
        raise ValueError("Invalid duration or amplitude range")

    rng = np.random.default_rng(seed)
    points_ms = []
    amplitudes_khz = []
    elapsed_ms = 0.0

    while onset_ms + elapsed_ms <= time_ms[-1]:
        elapsed_ms += rng.uniform(min_duration_ms, max_duration_ms)
        points_ms.append(elapsed_ms)
        amplitudes_khz.append(rng.uniform(min_amplitude_khz, max_amplitude_khz))

    return points_ms, amplitudes_khz


def _build_random_step(time_ms, baseline_khz, onset_ms, min_duration_ms,
                       max_duration_ms, min_amplitude_khz, max_amplitude_khz, seed=None):
    points, amplitudes = _random_stimulation(
        time_ms, onset_ms, min_duration_ms, max_duration_ms,
        min_amplitude_khz, max_amplitude_khz, seed
    )

    return _build_step(time_ms, baseline_khz, onset_ms,
                       [0.0, *points[:-1]], amplitudes)


def _build_random_smooth(time_ms, baseline_khz, onset_ms, min_duration_ms,
                         max_duration_ms, min_amplitude_khz, max_amplitude_khz,
                         seed=None):
    points, amplitudes = _random_stimulation(
        time_ms, onset_ms, min_duration_ms, max_duration_ms,
        min_amplitude_khz, max_amplitude_khz, seed
    )

    return _build_smooth(time_ms, baseline_khz, onset_ms, points, amplitudes)


STIMULUS_BUILDERS = {
    "pulse": _build_pulse,
    "pulse_train": _build_pulse_train,
    "step": _build_step,
    "smooth": _build_smooth,
    "random_step": _build_random_step,
    "random_smooth": _build_random_smooth,
}

NOISE_FIELDS = {"mean_khz", "std_khz", "noise_seed"}


def _add_gaussian_noise(signal, mean_khz=0.0, std_khz=0.0, noise_seed=None):
    if std_khz == 0:
        return signal
    rng = np.random.default_rng(noise_seed)
    return signal + rng.normal(mean_khz, std_khz, signal.shape)


def build_stimulus(stimulus, duration_ms, dt_ms):
    time_ms = np.arange(0.0, duration_ms, dt_ms)
    output = {}
    

    # ToDo: Por ahora solo se permite una señal de un tipo. 
    #       Crear tipos de estímulos que permitan combinaciones.
    for key, value in stimulus.items():
        if value is None: continue

        # Es un diccionario con un solo elemento, que indica el tipo de estímulo y sus parámetros.
        # ToDo: Permitir multiples elementos.        
        kind, parameters = next(iter(value.items()))

        if kind in STIMULUS_BUILDERS:            
            signal_parameters = {name: setting for name, setting in parameters.items()
                                 if name not in NOISE_FIELDS}
            noise_parameters = {name: setting for name, setting in parameters.items()
                                if name in NOISE_FIELDS}
            
            signal = STIMULUS_BUILDERS[kind](time_ms, **signal_parameters)
            output[key] = _add_gaussian_noise(signal, **noise_parameters)


    for key, value in stimulus.items():
        if value is None: continue

        # Ya sabemos que es un diccionario de un solo elemento.
        source, scale = next(iter(value.items())) 

        if source in stimulus:
            if source not in output:
                raise ValueError(
                    f"Stimulus value for {key} requires '{source}' to be defined."
                )

            output[key] = scale * output[source]

    return time_ms, [(key, output[key][:, np.newaxis]) for key in output]


In [ ]:
stimuli = {}
for condition, configurations in (("Without noise", STIMULUS_CONFIGS),
                                  ("With Gaussian noise", NOISE_STIMULUS_CONFIG)):
    stimuli[condition] = {}
    for name, stimulus_config in configurations.items():
        time_ms, stimulus = build_stimulus(stimulus_config, DURATION_MS, DT_MS)
        stimuli[condition][name] = stimulus
pprint('Stimulus configurations:', list(STIMULUS_CONFIGS))


## Deterministic Simulation


### Run simulation


In [ ]:
integrator = build_integrator(DT_MS, integrator_class="HeunDeterministic")
results = {}
for condition, configurations in stimuli.items():
    results[condition] = {}
    for name, stimulus in configurations.items():
        model = build_model(
            MODEL_PARAMETERS, INTEGER_PARAMETERS,
            model_class="ZerlautAdaptationFirstOrder",
            variable_of_interest=OUTPUT_STATES,
        )
        simulator = build_single_region_simulator(model, stimulus, integrator)
        time_sim, raw = simulator.run(n_steps=len(time_ms))[0]
        results[condition][name] = raw[:, :, 0, 0] # history, state, region, node,

        pprint(f"{condition}, {name}: {len(time_sim)} samples")


### Plot results


In [ ]:
metadata = ResultMetadata(
    simulation=SimulationMetadata(DURATION_MS, DT_MS, integrator, OUTPUT_DTYPE),
    model=ModelMetadata(
        name="ZerlautAdaptationFirstOrder", parameters=MODEL_PARAMETERS,
        integer_parameters=list(INTEGER_PARAMETERS), output_variables=OUTPUT_STATES,
        state_units=STATE_UNITS, stimulus_units=STIMULUS_UNITS,
    ),
    stimulus={},
)

# Dos figuras un con ruido y otra sin ruido.
for condition, condition_results in results.items():
    fig = plt.figure(figsize=(4.5 * len(condition_results), 6), constrained_layout=True)
    grid = fig.add_gridspec(2, len(condition_results))
    fig.suptitle(condition)


    for column, (name, states) in enumerate(condition_results.items()):
        stimulus_ax = fig.add_subplot(grid[0, column])

        # ToDo: Por ahora no se puede usar la función plot para dibuzar los estimulos!!!
        # esta función usa las funciones de construir estimulos que están en src!!!!
        for parameter, signal in stimuli[condition][name]:
            stimulus_ax.plot(time_ms, signal[:, 0], label=parameter)
        stimulus_ax.set_title(name)
        stimulus_ax.set_ylabel("Stimulus (kHz)")
        stimulus_ax.grid(alpha=0.25)
        stimulus_ax.legend()


        plot_results(time_sim, {"E": states[:, 0]}, metadata, show_stimulus=False,
                     ax=fig.add_subplot(grid[1, column]))
        
    plt.show()
